In this demo we perform demo DuckLake in a Jupyer notebook.

In [1]:
import duckdb
from pathlib import Path

In [2]:
DUCKLAKE_FOLDER = Path("../ducklake_basic")

### DELETE THIS SECTION IN BLOG: Prepare file system

In [3]:
import shutil

In [4]:
if not DUCKLAKE_FOLDER.exists():
    DUCKLAKE_FOLDER.mkdir()

In [5]:
# Delete the contents of the folder if anything exists
for item in DUCKLAKE_FOLDER.iterdir():
    if item.is_file() or item.is_symlink():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

### Install the DuckLake extension

In [6]:
duckdb.sql("INSTALL ducklake")

### Define the path to the metadata

In [7]:
ducklake_metadata = DUCKLAKE_FOLDER / "metadata"

### Define the path to the data files

In [8]:
ducklake_files = DUCKLAKE_FOLDER / "data_files"

### Attach to the DuckLake database

The deployment of a DuckLake database requires you to make basic choices about which technology you want to use for:

- **Metadata backends**: DuckDB, SQLite, PostgreSQL, MySQL, MotherDuck
- **Storage backends**: Local files, S3, Azure Blob Storage, Google Cloud Storage, Cloudflare R2

This provides a high degree of flexibility enabling you to use DuckLake from fully local setups to cloud-scale architectures and make the transition between the two trivial.

In this case we are using DuckDB as the metadata backend and the local file system for parquet format data files.

In [9]:
duckdb.sql(f"""
    ATTACH 'ducklake:{ducklake_metadata}' AS ducklake_basic_db (DATA_PATH '{ducklake_files}');
    USE ducklake_basic_db;
    """)


### Create schema

In [10]:
duckdb.sql("""
    CREATE SCHEMA IF NOT EXISTS retail_sales;
    USE retail_sales;
    """)

### Create tables

In the DuckLake documentation we learn that:

> DuckLake has limited support for constraints. The only constraint type that is currently supported is NOT NULL. It does not support PRIMARY KEY, FOREIGN KEY, UNIQUE or CHECK constraints.

Note the work **currently** in the statement above.  Perhaps support for more constraints is planned?

In response to the question "Are constraints such as primary keys and foreign keys supported?", the FAQ also states:

> No. Similar to other data lakehouse technologies, DuckLake does not support constraints, keys, or indexes.

So it appears that for now, the team are focusing on feature parity with other lakehouse technologies such as Iceberg and Delta.

If you try to use these features, DuckLake will raise a Not Implemented error.

In [11]:
try: duckdb.sql("""
    DROP TABLE IF EXISTS customer;
    CREATE TABLE IF NOT EXISTS customer (customer_id INTEGER, first_name VARCHAR, last_name VARCHAR, date_joined DATE, PRIMARY KEY (customer_id));
    """)
except Exception as e:
    print(e)

Not implemented Error: PRIMARY KEY/UNIQUE constraints are not supported in DuckLake


Let's continue without the primary key constraint.

In [12]:
duckdb.sql("""
    DROP TABLE IF EXISTS customer;
    CREATE TABLE IF NOT EXISTS customer (customer_id INTEGER NOT NULL, first_name VARCHAR, last_name VARCHAR, date_joined DATE);
    """)

### Insert data

Let's introduce some data to the `cusomter` table by insering three new rows.

In [13]:
duckdb.sql("""
    INSERT INTO customer (customer_id, first_name, last_name, date_joined) VALUES
    (1, 'Jane', 'Dunbar', '2023-01-11'),
    (2, 'Jimmy', 'Smith', '2024-08-26'),
    (3, 'Alice', 'Johnston', '2023-05-05');
    """)

In [14]:
duckdb.sql("""
    SELECT * FROM ducklake_basic_db.customer;
    """)

┌─────────────┬────────────┬───────────┬─────────────┐
│ customer_id │ first_name │ last_name │ date_joined │
│    int32    │  varchar   │  varchar  │    date     │
├─────────────┼────────────┼───────────┼─────────────┤
│           1 │ Jane       │ Dunbar    │ 2023-01-11  │
│           2 │ Jimmy      │ Smith     │ 2024-08-26  │
│           3 │ Alice      │ Johnston  │ 2023-05-05  │
└─────────────┴────────────┴───────────┴─────────────┘

### Inspect the raw data

We can see a single parquet file has been written.  We can load load this file independetly of DuckLake and see that it contains the 3 rows of raw data.

In [15]:
duckdb.sql(f"""
    FROM glob('{ducklake_files}/*');
    """)

┌────────────────────────────────────────────────────────────────────────────────────┐
│                                        file                                        │
│                                      varchar                                       │
├────────────────────────────────────────────────────────────────────────────────────┤
│ ../ducklake_basic/data_files/ducklake-0197a6c9-c10a-74fd-af2b-c4267f4c0897.parquet │
└────────────────────────────────────────────────────────────────────────────────────┘

In [16]:
# Extract the name of the first parquert file
parquet_file = duckdb.sql(f"""
    SELECT file FROM glob('{ducklake_files}/*.parquet') LIMIT 1;
    """).fetchone()[0]

print(f"Parquet file found: {parquet_file}")

Parquet file found: ../ducklake_basic/data_files/ducklake-0197a6c9-c10a-74fd-af2b-c4267f4c0897.parquet


In [17]:
# Inspect the contents of the parquet file
duckdb.sql(f"""
    SELECT * FROM read_parquet('{parquet_file}');
    """)

┌─────────────┬────────────┬───────────┬─────────────┐
│ customer_id │ first_name │ last_name │ date_joined │
│    int32    │  varchar   │  varchar  │    date     │
├─────────────┼────────────┼───────────┼─────────────┤
│           1 │ Jane       │ Dunbar    │ 2023-01-11  │
│           2 │ Jimmy      │ Smith     │ 2024-08-26  │
│           3 │ Alice      │ Johnston  │ 2023-05-05  │
└─────────────┴────────────┴───────────┴─────────────┘

### Delete data from table

Delete one of the customers from the table using their ID.

In [18]:
duckdb.sql("""
    DELETE  FROM ducklake_basic_db.customer WHERE customer_id = 2;
    """)

In [19]:
duckdb.sql("""
    SELECT * FROM ducklake_basic_db.customer;
    """)

┌─────────────┬────────────┬───────────┬─────────────┐
│ customer_id │ first_name │ last_name │ date_joined │
│    int32    │  varchar   │  varchar  │    date     │
├─────────────┼────────────┼───────────┼─────────────┤
│           1 │ Jane       │ Dunbar    │ 2023-01-11  │
│           3 │ Alice      │ Johnston  │ 2023-05-05  │
└─────────────┴────────────┴───────────┴─────────────┘

### Inspect how deletes are handled

The deletes are captured in specific Parquet file which references the parquet file from which the record(s) should be deleted and the position of the rows which should be deleted, we can inspect this file:

In [20]:
duckdb.sql(f"""
    FROM glob('{ducklake_files}/*-delete.parquet');
    """)

┌───────────────────────────────────────────────────────────────────────────────────────────┐
│                                           file                                            │
│                                          varchar                                          │
├───────────────────────────────────────────────────────────────────────────────────────────┤
│ ../ducklake_basic/data_files/ducklake-0197a6c9-c1f1-7e5e-b087-dafdbc40ece9-delete.parquet │
└───────────────────────────────────────────────────────────────────────────────────────────┘

In [21]:
# Extract the name of the first parquert file
parquet_file = duckdb.sql(f"""
    SELECT file FROM glob('{ducklake_files}/*delete.parquet') LIMIT 1;
    """).fetchone()[0]

We can see that the name of the file referenced below matchs the name of the file which contains the data for the `customer` table.

In [22]:
# Inspect the contents of the parquet file
duckdb.sql(f"""
    SELECT * FROM read_parquet('{parquet_file}');
    """)

┌────────────────────────────────────────────────────────────────────────────────────┬───────┐
│                                     file_path                                      │  pos  │
│                                      varchar                                       │ int64 │
├────────────────────────────────────────────────────────────────────────────────────┼───────┤
│ ../ducklake_basic/data_files/ducklake-0197a6c9-c10a-74fd-af2b-c4267f4c0897.parquet │     1 │
└────────────────────────────────────────────────────────────────────────────────────┴───────┘

### Update data

Let's update a single customer record.

In [23]:
duckdb.sql("""
    UPDATE customer SET first_name = 'Alice', last_name = 'Fraser' WHERE customer_id = 3;
    """)

### Multiple actions as a single atomic transaction

Here we create a new `email` column and then make a number of updates to existing customers to add their email addresses.

We perform these operations between a `BEGIN TRANSACTION` and `COMMIT` to implement these changes as a single atomic transaction.

In [24]:
duckdb.sql("""
    BEGIN TRANSACTION;
    ALTER TABLE customer ADD COLUMN email VARCHAR;
    UPDATE customer SET email = 'jane.dunbar@email.com' WHERE customer_id = 1;
    UPDATE customer SET email = 'james.smith@email.com' WHERE customer_id = 2;
    UPDATE customer SET email = 'alice.johnston.com' WHERE customer_id = 3;
    COMMIT
    """)

### View snapshots

Below we can see all of the operations we have performed so far:

- Created the schemas (`main` is created by default, we added a `retail_sales` schema) using a `CREATE SCHEMA`
- Created a `customer` table using a `CREATE TABLE`
- Inserted 3 rows via an `INSERT` operation
- Updated a row via an `UPDATE` operation
- Under a single transaction:
    - Upated the schema to add an `email` columns
    - Updated 3 existing rows to add the missing email address
- Deleted row via a `DELETE` operation

Notes:
- The schema_version evolves only when we make a schema related change.
- The `UPDATE` action appears as a delete and insert in the "changes".
- The set of actions carried out between `BEGIN TRANSACTION` and `COMMIT` appear as a single snapshot_id reflecting the fact that DuckLake supports ACID transactions.

In [25]:
duckdb.sql("""
    SELECT * FROM ducklake_snapshots('ducklake_basic_db');
    """)

┌─────────────┬────────────────────────────┬────────────────┬─────────────────────────────────────────────────────────────────────────┐
│ snapshot_id │       snapshot_time        │ schema_version │                                 changes                                 │
│    int64    │  timestamp with time zone  │     int64      │                         map(varchar, varchar[])                         │
├─────────────┼────────────────────────────┼────────────────┼─────────────────────────────────────────────────────────────────────────┤
│           0 │ 2025-06-25 11:12:09.282+00 │              0 │ {schemas_created=[main]}                                                │
│           1 │ 2025-06-25 11:12:09.383+00 │              1 │ {schemas_created=[retail_sales]}                                        │
│           2 │ 2025-06-25 11:12:09.441+00 │              2 │ {tables_created=[retail_sales.customer]}                                │
│           3 │ 2025-06-25 11:12:09.458+00 │    

### Inspect snapshots for single table

We can now view all of the incremental changes that have been made to the table.  DuckLake provides powerful capabilities for understanding what changed between versions.

Here we are using the `ducklake_table_changes` function.  It takes as input the database, schema and table for which changes should be returned, and two bounds: the start snapshot and the end snapshot (inclusive). The bounds can be given either as a snapshot id, or as a timestamp.

The result of the function is the set of changes, read using the table schema as of the end snapshot provided, and three extra columns: snapshot_id, rowid and change_type.

It feels a bit odd that the `ducklake_table_changes` doesn't enable a wild card option for the start snapshot and end snapshot.  Here we are setting these values to the min and max snapshot_id from the `ducklake_snapshots` above to get a full history for the table up to present time.

In [26]:
# Get the min and max snapshot_id
min_snapshot_id = duckdb.sql("""
    SELECT MIN(snapshot_id) AS min_snapshot_id FROM ducklake_snapshots('ducklake_basic_db');
    """).fetchone()[0]

max_snapshot_id = duckdb.sql("""
    SELECT MAX(snapshot_id) AS max_snapshot_id FROM ducklake_snapshots('ducklake_basic_db');
    """).fetchone()[0]

print(f"Min snapshot ID: {min_snapshot_id}, Max snapshot ID: {max_snapshot_id}")

Min snapshot ID: 0, Max snapshot ID: 6


In [27]:
duckdb.sql(f"""
    SELECT * FROM ducklake_table_changes('ducklake_basic_db', 'retail_sales', 'customer', {min_snapshot_id}, {max_snapshot_id})
    ORDER BY snapshot_id ASC, rowid ASC, change_type DESC;
    """)

┌─────────────┬───────┬──────────────────┬─────────────┬────────────┬───────────┬─────────────┬───────────────────────┐
│ snapshot_id │ rowid │   change_type    │ customer_id │ first_name │ last_name │ date_joined │         email         │
│    int64    │ int64 │     varchar      │    int32    │  varchar   │  varchar  │    date     │        varchar        │
├─────────────┼───────┼──────────────────┼─────────────┼────────────┼───────────┼─────────────┼───────────────────────┤
│           3 │     0 │ insert           │           1 │ Jane       │ Dunbar    │ 2023-01-11  │ NULL                  │
│           3 │     1 │ insert           │           2 │ Jimmy      │ Smith     │ 2024-08-26  │ NULL                  │
│           3 │     2 │ insert           │           3 │ Alice      │ Johnston  │ 2023-05-05  │ NULL                  │
│           4 │     1 │ delete           │           2 │ Jimmy      │ Smith     │ 2024-08-26  │ NULL                  │
│           5 │     2 │ update_preimage 

### Time travel

Let's re-wind back one snapshot_id (view the table before the `DELETE` operation).

In [28]:
duckdb.sql(f"""
    SELECT * FROM customer AT (VERSION => {max_snapshot_id - 1});
    """)

┌─────────────┬────────────┬───────────┬─────────────┐
│ customer_id │ first_name │ last_name │ date_joined │
│    int32    │  varchar   │  varchar  │    date     │
├─────────────┼────────────┼───────────┼─────────────┤
│           1 │ Jane       │ Dunbar    │ 2023-01-11  │
│           3 │ Alice      │ Fraser    │ 2023-05-05  │
└─────────────┴────────────┴───────────┴─────────────┘

### Add new table

We are now going to a new orders table and populate it with transactions.

In [29]:
duckdb.sql("""
    USE ducklake_basic_db;
    USE retail_sales;
    """)

In [30]:
duckdb.sql("""
    DROP TABLE IF EXISTS orders;
    CREATE TABLE IF NOT EXISTS orders (order_id INTEGER, customer_id INTEGER, order_date DATE, product_id INTEGER, product_name VARCHAR, amount DECIMAL(10, 2));
    """)

In [31]:
duckdb.sql("""
    INSERT INTO orders (order_id, customer_id, product_id, product_name, order_date, amount) VALUES
    (1, 1, 101, 'Widget A', '2023-01-15', 19.50),
    (2, 1, 102, 'Widget B', '2023-01-20', 29.99),
    (3, 3, 103, 'Widget A', '2023-02-10', 19.50);
""")

In [32]:
duckdb.sql("""
    SELECT * FROM orders;
    """)

┌──────────┬─────────────┬────────────┬────────────┬──────────────┬───────────────┐
│ order_id │ customer_id │ order_date │ product_id │ product_name │    amount     │
│  int32   │    int32    │    date    │   int32    │   varchar    │ decimal(10,2) │
├──────────┼─────────────┼────────────┼────────────┼──────────────┼───────────────┤
│        1 │           1 │ 2023-01-15 │        101 │ Widget A     │         19.50 │
│        2 │           1 │ 2023-01-20 │        102 │ Widget B     │         29.99 │
│        3 │           3 │ 2023-02-10 │        103 │ Widget A     │         19.50 │
└──────────┴─────────────┴────────────┴────────────┴──────────────┴───────────────┘

### Maintain consistency across multiple tables using a transaction

We have some new order data for a new customer.  We want to protect the integrity of the database, so we make between a `BEGIN TRANSACTION` and `COMMIT` to make the updates to both the `customer` and `orders` tables in a single atomic operation.

This level of granular change tracking, combined with ACID transactions across multiple tables, represents a significant advancement over existing lakehouse formats that struggle with consistent multi-table operations.

In [33]:
duckdb.sql("""
    BEGIN TRANSACTION;
    INSERT INTO customer (customer_id, first_name, last_name, date_joined) VALUES
    (4, 'Bob', 'Brown', '2023-03-01');
    INSERT INTO orders (order_id, customer_id, product_id, product_name, order_date, amount) VALUES
    (4, 4, 104, 'Widget B', '2023-03-05', 29.99),
    (5, 4, 105, 'Widget C', '2023-02-15', 59.99),
    (6, 4, 106, 'Widget A', '2023-01-25', 19.50);
    COMMIT;
""")

In [34]:
duckdb.sql("""
    SELECT * FROM ducklake_snapshots('ducklake_basic_db');
    """)

┌─────────────┬────────────────────────────┬────────────────┬─────────────────────────────────────────────────────────────────────────┐
│ snapshot_id │       snapshot_time        │ schema_version │                                 changes                                 │
│    int64    │  timestamp with time zone  │     int64      │                         map(varchar, varchar[])                         │
├─────────────┼────────────────────────────┼────────────────┼─────────────────────────────────────────────────────────────────────────┤
│           0 │ 2025-06-25 11:12:09.282+00 │              0 │ {schemas_created=[main]}                                                │
│           1 │ 2025-06-25 11:12:09.383+00 │              1 │ {schemas_created=[retail_sales]}                                        │
│           2 │ 2025-06-25 11:12:09.441+00 │              2 │ {tables_created=[retail_sales.customer]}                                │
│           3 │ 2025-06-25 11:12:09.458+00 │    

In [35]:
# Get new max snapshot_id
max_snapshot_id = duckdb.sql("""
    SELECT MAX(snapshot_id) AS max_snapshot_id FROM ducklake_snapshots('ducklake_basic_db');
    """).fetchone()[0]

print(f"Max snapshot ID: {max_snapshot_id}")

Max snapshot ID: 9


In [36]:
duckdb.sql(f"""
    SELECT * FROM ducklake_table_changes('ducklake_basic_db', 'retail_sales', 'customer', {max_snapshot_id}, {max_snapshot_id})
    ORDER BY snapshot_id;
    """)

┌─────────────┬───────┬─────────────┬─────────────┬────────────┬───────────┬─────────────┬─────────┐
│ snapshot_id │ rowid │ change_type │ customer_id │ first_name │ last_name │ date_joined │  email  │
│    int64    │ int64 │   varchar   │    int32    │  varchar   │  varchar  │    date     │ varchar │
├─────────────┼───────┼─────────────┼─────────────┼────────────┼───────────┼─────────────┼─────────┤
│           9 │     6 │ insert      │           4 │ Bob        │ Brown     │ 2023-03-01  │ NULL    │
└─────────────┴───────┴─────────────┴─────────────┴────────────┴───────────┴─────────────┴─────────┘

In [37]:
duckdb.sql(f"""
    SELECT * FROM ducklake_table_changes('ducklake_basic_db', 'retail_sales', 'orders', {max_snapshot_id}, {max_snapshot_id})
    ORDER BY snapshot_id;
    """)

┌─────────────┬───────┬─────────────┬──────────┬─────────────┬────────────┬────────────┬──────────────┬───────────────┐
│ snapshot_id │ rowid │ change_type │ order_id │ customer_id │ order_date │ product_id │ product_name │    amount     │
│    int64    │ int64 │   varchar   │  int32   │    int32    │    date    │   int32    │   varchar    │ decimal(10,2) │
├─────────────┼───────┼─────────────┼──────────┼─────────────┼────────────┼────────────┼──────────────┼───────────────┤
│           9 │     3 │ insert      │        4 │           4 │ 2023-03-05 │        104 │ Widget B     │         29.99 │
│           9 │     4 │ insert      │        5 │           4 │ 2023-02-15 │        105 │ Widget C     │         59.99 │
│           9 │     5 │ insert      │        6 │           4 │ 2023-01-25 │        106 │ Widget A     │         19.50 │
└─────────────┴───────┴─────────────┴──────────┴─────────────┴────────────┴────────────┴──────────────┴───────────────┘